## Red dot tracker

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os

# === Folder containing videos ===
folder_path = r"C:\Users\David\Videos"

# === Define target color range in HSV ===
# Red wraps around the hue range, so we combine two masks.
lower_red1 = np.array([0, 70, 70])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 70, 70])
upper_red2 = np.array([180, 255, 255])

# === Supported video formats ===
video_exts = ('.mov', '.mp4', '.avi', '.mkv', '.wmv')

# === Helper: click event callback ===
def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        param["point"] = (x, y)
        print(f"Reference point selected: {param['point']}")
        cv2.destroyWindow("Select Reference Point")

# === Loop through all files ===
for filename in os.listdir(folder_path):
    if not filename.lower().endswith(video_exts):
        continue

    video_path = os.path.join(folder_path, filename)
    output_csv = os.path.splitext(video_path)[0] + "_positions.csv"

    print(f"\nProcessing: {filename}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open {filename}")
        continue

    # === Show first frame for manual reference selection ===
    ret, first_frame = cap.read()
    if not ret:
        print(f"❌ Cannot read {filename}")
        cap.release()
        continue

    # --- Reset click container for each video ---
    click_data = {"point": None}

    # Ask user to click the reference (black dot)
    cv2.imshow("Select Reference Point", first_frame)
    cv2.setMouseCallback("Select Reference Point", click_event, click_data)
    print("🖱️ Click the black reference dot in the window.")

    # Wait for user to click
    while click_data["point"] is None:
        if cv2.waitKey(20) & 0xFF == 27:  # Escape to skip
            break
    ref_point = click_data["point"]
    if ref_point is None:
        print("⚠️ No reference selected — skipping this video.")
        cap.release()
        cv2.destroyAllWindows()
        continue

    ref_x, ref_y = ref_point
    cv2.destroyAllWindows()
    print(f"✅ Reference point for {filename}: ({ref_x}, {ref_y})")

    # Reset to start
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    positions = []
    frame_idx = 0

    # === Process frames ===
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Threshold for red (combine two ranges)
        mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
        mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
        mask = cv2.bitwise_or(mask1, mask2)

        # Clean up noise
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            c = max(contours, key=cv2.contourArea)
            M = cv2.moments(c)
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                dx = cx - ref_x
                dy = cy - ref_y
                positions.append((frame_idx, cx, cy, dx, dy))
            else:
                positions.append((frame_idx, np.nan, np.nan, np.nan, np.nan))
        else:
            positions.append((frame_idx, np.nan, np.nan, np.nan, np.nan))

    cap.release()

    # === Save CSV ===
    df = pd.DataFrame(positions, columns=["frame", "x", "y", "dx_from_ref", "dy_from_ref"])
    df.to_csv(output_csv, index=False)
    print(f"✅ Saved: {output_csv}")

print("\n🎉 All videos processed!")



Processing: spring_m3_tracked.mp4
🖱️ Click the black reference dot in the window and press any key after selecting.
Reference point selected: (894, 536)
✅ Reference point: (894, 536)
✅ Saved: C:\Users\David\Videos\spring_m3_tracked_positions.csv

Processing: uw_m1_tracked.mp4
🖱️ Click the black reference dot in the window and press any key after selecting.
✅ Reference point: (894, 536)
✅ Saved: C:\Users\David\Videos\uw_m1_tracked_positions.csv

Processing: uw_m2_tracked.mp4
🖱️ Click the black reference dot in the window and press any key after selecting.
✅ Reference point: (894, 536)
✅ Saved: C:\Users\David\Videos\uw_m2_tracked_positions.csv

Processing: uw_m3_tracked.mp4
🖱️ Click the black reference dot in the window and press any key after selecting.
✅ Reference point: (894, 536)
✅ Saved: C:\Users\David\Videos\uw_m3_tracked_positions.csv
🎉 All videos processed!


## Green dot tracker

In [3]:
import cv2
import numpy as np
import pandas as pd
import os

# === Folder containing videos ===
folder_path = r"C:\Users\David\Videos"

# === Define target color range in HSV ===
# For green (~#00ff00)
# You can fine-tune these if needed
lower_hsv = np.array([40, 70, 70])
upper_hsv = np.array([80, 255, 255])

# === Supported video formats ===
video_exts = ('.mov', '.mp4', '.avi', '.mkv', '.wmv')

# === Helper: click event callback ===
def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        param["point"] = (x, y)
        print(f"Reference point selected: {param['point']}")
        cv2.destroyWindow("Select Reference Point")

# === Loop through all files ===
for filename in os.listdir(folder_path):
    if not filename.lower().endswith(video_exts):
        continue

    video_path = os.path.join(folder_path, filename)
    output_csv = os.path.splitext(video_path)[0] + "_positions.csv"

    print(f"\nProcessing: {filename}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open {filename}")
        continue

    # === Show first frame for manual reference selection ===
    ret, first_frame = cap.read()
    if not ret:
        print(f"❌ Cannot read {filename}")
        cap.release()
        continue

    # --- Reset click container for each video ---
    click_data = {"point": None}

    # Ask user to click the reference (black dot)
    cv2.imshow("Select Reference Point", first_frame)
    cv2.setMouseCallback("Select Reference Point", click_event, click_data)
    print("🖱️ Click the black reference dot in the window.")

    # Wait for user to click
    while click_data["point"] is None:
        if cv2.waitKey(20) & 0xFF == 27:  # Escape to skip
            break
    ref_point = click_data["point"]
    if ref_point is None:
        print("⚠️ No reference selected — skipping this video.")
        cap.release()
        cv2.destroyAllWindows()
        continue

    ref_x, ref_y = ref_point
    cv2.destroyAllWindows()
    print(f"✅ Reference point for {filename}: ({ref_x}, {ref_y})")

    # Reset to start
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    positions = []
    frame_idx = 0

    # === Process frames ===
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Threshold for green
        mask = cv2.inRange(hsv, lower_hsv, upper_hsv)

        # Clean up noise
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            c = max(contours, key=cv2.contourArea)
            M = cv2.moments(c)
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                dx = cx - ref_x
                dy = cy - ref_y
                positions.append((frame_idx, cx, cy, dx, dy))
            else:
                positions.append((frame_idx, np.nan, np.nan, np.nan, np.nan))
        else:
            positions.append((frame_idx, np.nan, np.nan, np.nan, np.nan))

    cap.release()

    # === Save CSV ===
    df = pd.DataFrame(positions, columns=["frame", "x", "y", "dx_from_ref", "dy_from_ref"])
    df.to_csv(output_csv, index=False)
    print(f"✅ Saved: {output_csv}")

print("\n🎉 All videos processed!")


Processing: M3T1_fixed.mp4
🖱️ Click the black reference dot in the window.
Reference point selected: (972, 492)
✅ Reference point for M3T1_fixed.mp4: (972, 492)
✅ Saved: C:\Users\David\Videos\M3T1_fixed_positions.csv

🎉 All videos processed!
